# Generate IDRs
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/v1/cookbook/notebooks/01_generate_idrs.ipynb)

Generate standalone IDRs and contextual replacements; export FASTA and sequence summaries.

This notebook runs independently. Select **Runtime → Change runtime type → GPU** in Colab.
First use downloads model weights. A GPU is recommended; CPU inference is supported but slower. Reduce sample counts and batch size for a first run.
The setup installs the `v1` release when IDiom is absent. If using an older installation,
upgrade to that release and restart the kernel. No adjacent helper files are required.

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "idiom[cookbook] @ git+https://github.com/rotskoff-group/idiom.git@v1"])
if importlib.util.find_spec("pandas") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pandas>=2"])

import json
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from idiom import IDiom, IDiomSAE
from idiom.data.records import Record
from idiom.utils.notebook_helpers import (
    load_inputs, idr_sequence, isolated, check_context, summaries, write_fasta,
    save_run, sequence_metrics, nearest_reference, split_records, example_file,
)
print("Python:", sys.version.split()[0])
started = time.perf_counter()

## Inputs and settings
Run top to bottom. Upload a FASTA using Colab's Files pane and set its path below, or
leave the demo input unchanged. `INPUT_MODE="idr"` accepts ordinary headers for isolated
IDRs; `"annotated"` requires full-protein headers ending in `_IDR_x-y` (1-based inclusive).
Python coordinates are 0-based, end-exclusive. These workflows do not predict IDR boundaries.

For persistent outputs, optionally mount Drive in your own cell with
`from google.colab import drive; drive.mount("/content/drive")`, then set `OUT_DIR` there.
Use a new output directory for each experiment. Rejected records are reported in an audit.

In [ ]:
MODEL_ID = "jxliu2/idiom-20M"
DEVICE = "auto"
N = 8
BATCH_SIZE = 4
SEED = 0
TEMPERATURE = 1.0
TOP_P = 0.95
LENGTH_RANGE = (20, 60)
MAX_NEW_TOKENS = 128
OUT_DIR = Path("generation_outputs")
# Optional: annotated full-protein FASTA; first accepted record is redesigned.
INPUT_FASTA = None
INPUT_MODE = "annotated"
MAX_RECORDS = 1


In [ ]:
started = time.perf_counter()


## Generate standalone IDRs

`length_range` filters oversampled candidates, so fewer than `N` may be returned. This is not
fixed-length generation. A seed is reproducible for fixed settings including batch size.
These candidates require downstream evaluation; generation alone does not establish disorder
or function. Lower `BATCH_SIZE` if GPU memory is limited.


In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
model = IDiom.from_pretrained(MODEL_ID, device=DEVICE)
kwargs = dict(n=N, batch_size=BATCH_SIZE, seed=SEED, temperature=TEMPERATURE,
              top_p=TOP_P, length_range=LENGTH_RANGE, max_new_tokens=MAX_NEW_TOKENS)
idrs = model.generate_unprompted(**kwargs)
generated = [Record(f"generated_{i}", s, 0, len(s)) for i, s in enumerate(idrs) if s]
write_fasta(generated, OUT_DIR / "idrs.fasta")
print(f"Requested {N}; retained {len(generated)} nonempty IDRs")
display(pd.DataFrame([dict(record_id=r.accession, sequence=r.full_seq, length=len(r.full_seq)) for r in generated]))


## Redesign an annotated region

The default is the bundled HP1α example (IDR residues 79–123). For your own protein, upload an
annotated FASTA and set `INPUT_FASTA`. Flanks are retained unchanged; generated IDRs can have
different lengths, so the exported coordinates are updated. The original IDR is saved as a
reference. Only the first valid record is used here; the package's `generate_prompted_fasta`
method supports batch workflows.


In [ ]:
prompt_path = INPUT_FASTA
if prompt_path is None:
    prompt_path = example_file("prompted_grpo/P45973.fasta", OUT_DIR / "inputs")
records, audit = load_inputs(prompt_path, INPUT_MODE, MAX_RECORDS)
audit.to_csv(OUT_DIR / "input_audit.csv", index=False)
display(audit)
if not records:
    raise ValueError("No accepted annotated proteins.")
record = records[0]
check_context(records, model.model.cfg.max_seq_len, include_flanks=True)
print("Original IDR:", idr_sequence(record))
replacement = model.generate_prompted(record.full_seq, record.idr_start, record.idr_end, **kwargs)
redesigned = []
for i, s in enumerate(replacement):
    if s:
        full = record.full_seq[:record.idr_start] + s + record.full_seq[record.idr_end:]
        redesigned.append(Record(f"{record.accession}_design_{i}", full, record.idr_start, record.idr_start + len(s)))
write_fasta(redesigned, OUT_DIR / "redesigned_proteins.fasta")
write_fasta(isolated(redesigned), OUT_DIR / "redesigned_idrs.fasta")
write_fasta(isolated(records[:1]), OUT_DIR / "original_idr.fasta")
print(f"Retained {len(redesigned)} prompted replacements")


## Inspect and export candidates

The table connects each prompted design to its input record. Compare sequence length and
composition first; use notebook 02 for embeddings and notebook 03 for SAE feature interpretation. Empty outputs remain valid files but cannot be analyzed downstream.


In [ ]:
rows = [dict(record_id=r.accession, source_record=None, kind="unprompted", sequence=idr_sequence(r),
             length=len(idr_sequence(r))) for r in generated]
rows += [dict(record_id=r.accession, source_record=record.accession, kind="prompted",
              sequence=idr_sequence(r), length=len(idr_sequence(r))) for r in redesigned]
candidates = pd.DataFrame(rows, columns=["record_id", "source_record", "kind", "sequence", "length"])
metrics = sequence_metrics(candidates.sequence.tolist())
candidates["entropy"] = metrics.entropy
candidates["charged_fraction"] = metrics.charged_fraction
candidates["duplicate"] = metrics.duplicate
candidates.to_csv(OUT_DIR / "candidates.csv", index=False)
display(candidates)
if len(candidates):
    fig, ax = plt.subplots(figsize=(6, 3), constrained_layout=True)
    for label, group in candidates.groupby("kind"):
        ax.hist(group.length, bins=10, alpha=0.5, label=label)
    ax.set(xlabel="IDR length", ylabel="Candidate count")
    ax.legend()
    fig.savefig(OUT_DIR / "candidate_lengths.png", dpi=160)
    plt.show()
save_run(OUT_DIR, dict(model=MODEL_ID, device=str(model.device), input=prompt_path, **kwargs), elapsed=time.perf_counter() - started)
print(f"Elapsed including model load: {time.perf_counter() - started:.1f} s")


## Save and continue

Keep the input audit and run settings with your exports. Continue with [SAE interpretation](03_interpret_sae_features.ipynb) or [fine-tuning](05_finetune_and_generate.ipynb).

## Download results
The archive includes tables, plots, inputs, and run settings.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR.resolve()), "zip", OUT_DIR)
print("Results:", OUT_DIR.resolve(), "\nDownload:", archive)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(archive)